# Estatística com Python: resumindo e analisando dados
Tipos de variáveis, distribuições de frequência, medidas de tendência central, separatrizes e de dispersão aplicadas a demandas de uma rede de varejo (vendas do e-commerce e remuneração dos colaboradores). No final, os desafios "Mão na Massa" de cada aula, resolvidos com uma amostra da PNAD 2015.

**Origem:** Alura, curso "Estatística com Python: resumindo e analisando dados".

**Dados:** lidos por URL do repositório do curso (`vendas_ecommerce.csv`, `colaboradores.csv` e `dados_desafio.csv` em https://github.com/alura-cursos/Estatisticas-Python-frequencias-medidas).

**Índice**
1. Entendendo os dados (Aula 01)
2. Identificando o perfil do público (Aula 02)
3. Analisando a tendência dos dados (Aula 03)
4. Investigando os dados dos colaboradores (Aula 04)
5. Analisando as variações dos dados (Aula 05)
6. Desafios (Mão na Massa por aula, PNAD 2015)

In [ ]:
# Setup: todas as bibliotecas já vêm instaladas no Colab
import math

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

## 1. Entendendo os dados
Somos **cientistas de dados** prestando consultoria a uma grande rede de varejo. Há duas bases: as **vendas do e-commerce** (perfil da clientela, para marketing e vendas) e a **remuneração dos colaboradores** (salários e equidade, para RH e financeiro). O desafio é aplicar distribuições de frequência, medidas de tendência central, separatrizes, de dispersão e visualização de dados para responder às demandas da empresa.

### 1.1 Lendo os dados de vendas

In [ ]:
url = 'https://raw.githubusercontent.com/alura-cursos/Estatisticas-Python-frequencias-medidas/refs/heads/main/dados/vendas_ecommerce.csv'
df = pd.read_csv(url)
df.shape

In [ ]:
df.info()

Para cada análise, duas perguntas importam: **qual o tipo de dado que tenho?** e **o que desejo investigar com ele?** A forma de análise depende do tipo e do comportamento do dado, no tripé da ciência de dados: **estatística**, **programação** e **conhecimento do negócio**.

### 1.2 Tipos de variáveis
![tipos de variáveis](https://github.com/afonsosr2/estatistica-r-frequencias-medidas/blob/main/imagens/outros/tipos_variaveis.png?raw=true)

- **Qualitativas (categóricas):** representam características ou categorias (ex.: sexo biológico e região).
  - **Nominais:** sem ordem natural (ex.: tipo de produto).
  - **Ordinais:** com ordem ou hierarquia (ex.: níveis de escolaridade).
- **Quantitativas (numéricas):** representam números mensuráveis, com significado matemático (ex.: idade e total da compra).
  - **Discretas:** valores inteiros e contáveis (ex.: quantidade de produtos).
  - **Contínuas:** qualquer valor em um intervalo (ex.: altura de pessoas).

Exemplo de **variável qualitativa nominal**: quantidade de vendas por categoria.

In [ ]:
cat_prod = df['categoria_produto'].value_counts().reset_index()
cat_prod

In [ ]:
plt.barh(cat_prod['categoria_produto'], cat_prod['count'])
plt.show()

### 1.3 Manipulando dados qualitativos ordinais
A `avaliacao` do produto é numérica, mas pode ser interpretada como **qualitativa ordinal**: as notas representam uma ordem de satisfação, de Péssimo (1) a Ótimo (5).

In [ ]:
sorted(df['avaliacao'].unique())

In [ ]:
df['avaliacao indicador'] = pd.Categorical(
    df['avaliacao'],
    categories=[1, 2, 3, 4, 5],
    ordered=True
)
labels = {1: 'Péssimo', 2: 'Ruim', 3: 'Regular', 4: 'Bom', 5: 'Ótimo'}
df['avaliacao indicador'] = df['avaliacao indicador'].map(labels)

df[['avaliacao', 'avaliacao indicador']].drop_duplicates().sort_values('avaliacao')

### 1.4 Diferenciando dados discretos de contínuos
Algumas variáveis numéricas podem ser encaixadas em mais de uma subdivisão. `quantidade`, `idade_cliente` e `tempo_entrega` são contagens (discretas); `preco_unitario` e `total_compra` são valores monetários em escala contínua.

In [ ]:
print('quantidade:', sorted(df['quantidade'].unique()))
print(f"Tivemos vendas a partir de R$ {df['total_compra'].min():,.2f} até R$ {df['total_compra'].max():,.2f}")
df[['preco_unitario', 'total_compra']].describe()

## 2. Identificando o perfil do público
Análises do perfil do público para o time de marketing e vendas, usando a **distribuição de frequência**, que organiza e resume dados em grupos ou classes.

### 2.1 Distribuição de frequências de dados qualitativos - 1 variável
**Demanda 1: acompanhamento das avaliações dos clientes.** A empresa quer acompanhar as avaliações (Péssimo, Ruim, Regular, Bom, Ótimo) para priorizar ações: melhorar produtos com mais feedbacks negativos e explorar os pontos fortes apontados pelos clientes satisfeitos.

A **distribuição de frequência** é uma tabela com os dados de acordo com a sua **frequência** ($f_i$ = quantidade de vezes que um valor ocorre). Serve para dados qualitativos e quantitativos. Para construir: ordenar os valores, determinar a frequência de cada valor e agrupar em classes ou intervalos.

**Frequência absoluta ($f_i$):** número de vezes em que uma observação aparece; a soma é o total de registros.

$$F = \sum_{i=1}^{n}f_i = f_1 + f_2 + ... + f_n$$

In [ ]:
freq_avaliacoes = (df.groupby('avaliacao indicador', observed=False)
                     .size()
                     .reset_index(name='freq. abs.')
                     .sort_values(by='avaliacao indicador'))

**Frequência relativa ($f_{ri}$):** razão entre a frequência absoluta de cada classe e a frequência total (multiplicada por 100 para percentual).

$$f_{ri} = \frac{f_i}{F}$$

In [ ]:
freq_avaliacoes['freq_rel'] = round(freq_avaliacoes['freq. abs.'] / freq_avaliacoes['freq. abs.'].sum() * 100, 1)
freq_avaliacoes.columns = ['Avaliacao', 'Absoluta', 'Relativa']
freq_avaliacoes

In [ ]:
plt.figure(figsize=(10, 6))
sns.barplot(data=freq_avaliacoes, x='Avaliacao', y='Absoluta')

plt.title('Distribuição de Frequências das Avaliações')
plt.xlabel('Avaliação')
plt.ylabel('Frequência')

# Rótulos com a frequência absoluta e o percentual
for index, row in freq_avaliacoes.iterrows():
    plt.text(index, row['Absoluta'] + 0.1, f"{row['Absoluta']} ({row['Relativa']:.1f}%)",
             ha='center', va='bottom', fontsize=12)

plt.show()

Há mais avaliações positivas ("Ótimo" e "Bom") que negativas ("Péssimo" e "Ruim"), mas o número de "Péssimo" está acima das avaliações neutras, um comportamento que o marketing pode investigar.

### 2.2 Distribuição de frequências de dados qualitativos - 2 variáveis
**Demanda 2: avaliações dos clientes por região.** Qual região tem mais avaliações positivas? Alguma concentra a maior parte das negativas? Há diferença que indique ações específicas por região?

A **tabela de contingência** (tabela cruzada ou distribuição de frequência bivariada) resume a relação entre variáveis categóricas divididas em grupos mutuamente exclusivos. Vantagens: organiza os dados de forma compacta, permite análises rápidas e serve de base para testes de independência entre variáveis categóricas.

In [ ]:
tab_avaliacoes_regiao = pd.crosstab(df['avaliacao indicador'], df['regiao_cliente'])
tab_avaliacoes_regiao

In [ ]:
av_relativa = round(pd.crosstab(df['avaliacao indicador'], df['regiao_cliente'], normalize='columns') * 100, 1)
av_relativa

**Qual região tem mais avaliações positivas?** Considerando "Ótimo" e "Bom", Centro-Oeste e Nordeste dividem o ranking, com mais de 55% de satisfação.

In [ ]:
rel_pos = av_relativa[av_relativa.index.isin(['Ótimo', 'Bom'])]
rel_pos.sum()

**Alguma região concentra a maior parte das avaliações negativas?** O Sul tem a maior porcentagem de avaliações negativas (34,3%), sendo a única com quase 20% das notas "Péssimo".

In [ ]:
rel_neg = av_relativa[av_relativa.index.isin(['Ruim', 'Péssimo'])]
rel_neg.sum()

**Há necessidade de ações específicas?** Sim: investigar a baixa avaliação no Sul; Sudeste e Norte concentram cada uma mais de 20% de notas "Regular".

### 2.3 Cruzando os dados dos clientes
**Demanda 3: perfil dos clientes.** Como o **ticket médio** varia entre o **sexo biológico** e a **região** da clientela? Para isso, a tabela cruzada recebe uma estatística (a média) de uma variável quantitativa.

In [ ]:
ticket_medio = round(pd.crosstab(df['sexo_biologico'], df['regiao_cliente'], values=df['total_compra'], aggfunc='mean'), 2)
ticket_medio

Distribuição dos clientes por sexo, no total e dentro de cada região:

In [ ]:
perfil_freq = df.groupby('sexo_biologico').size().reset_index(name='freq. abs.')
perfil_freq['freq. rel.'] = round(perfil_freq['freq. abs.'] / perfil_freq['freq. abs.'].sum() * 100, 1)
perfil_freq

In [ ]:
round(pd.crosstab(df['sexo_biologico'], df['regiao_cliente'], normalize='columns') * 100, 1)

O sexo masculino tem ticket médio acima do feminino no Centro-Oeste e no Nordeste (diferença próxima de R\$ 150,00); o feminino gasta mais no Sul e no Sudeste (diferença por volta de R\$ 300,00).

## 3. Analisando a tendência dos dados
As **medidas de tendência central** (média, mediana e moda) resumem os dados em um único valor.

### 3.1 Média
**Demanda 4: tempo de entrega por tipo de produto.** Calcular o **tempo médio de entrega por categoria**, para dar estimativas mais precisas no site e ajustar a logística.

A **média aritmética** é a soma de todos os valores dividida pelo número de observações; é um ponto de equilíbrio dos dados.

$$\mu = \frac 1n\sum_{i=1}^{n}X_i = \frac{X_1 + X_2 + ... + X_n}{n}$$

onde $n$ = número de observações e $X_i$ = valor da i-ésima observação.

In [ ]:
df['tempo_entrega'].mean()

In [ ]:
media_entrega = df.groupby('categoria_produto')['tempo_entrega'].mean().reset_index().round(1)
media_entrega.columns = ['Categoria', 'Média tempo de entrega']
media_entrega = media_entrega.sort_values(by='Média tempo de entrega', ascending=False)
media_entrega

In [ ]:
media = media_entrega['Média tempo de entrega'].mean()

plt.figure(figsize=(8, 6))
sns.barplot(data=media_entrega, x='Média tempo de entrega', y='Categoria')
plt.axvline(x=media, color='red')
plt.xlabel('Tempo de Entrega')
plt.ylabel('Categoria de Produto')
plt.show()

**A média é sempre a melhor forma de resumir os dados?** Não: por ser um ponto de equilíbrio, ela é muito influenciada pelos extremos. Aliada a outras medidas, ajuda a decidir se o dado precisa de tratamento ou transformação.

In [ ]:
resumo_tendencia_central = {
    'média': df['tempo_entrega'].mean(),
    'moda': df['tempo_entrega'].mode()[0],
    'mediana': df['tempo_entrega'].median()
}
resumo_tendencia_central

As três medidas da idade dos clientes por sexo biológico:

In [ ]:
df.groupby('sexo_biologico')['idade_cliente'].agg(
    media='mean',
    mediana='median',
    moda=lambda x: x.mode().iloc[0]
).round(0)

In [ ]:
sns.histplot(bins=10, data=df, x='idade_cliente', hue='sexo_biologico', kde=True, kde_kws={'bw_adjust': 2})
plt.show()

In [ ]:
plt.figure(figsize=(10, 8))
sns.set_style('darkgrid')
sns.histplot(df['idade_cliente'], bins=15, color='steelblue', alpha=0.3)

media = round(df['idade_cliente'].mean())
moda = round(df['idade_cliente'].mode()[0])
mediana = round(df['idade_cliente'].median())

plt.axvline(media, color='red', linestyle='--', label=f'Média: {media:.2f}')
plt.axvline(mediana, color='green', linestyle='-.', label=f'Mediana: {mediana:.2f}')
plt.axvline(moda, color='black', linestyle=':', label=f'Moda: {moda:.2f}')

plt.title('Distribuição das idades dos clientes', fontsize=22, ha='center')
plt.xlabel('Idade', fontsize=16)
plt.ylabel('Frequência', fontsize=16)
plt.legend(title='', fontsize=14)
plt.show()

Com o tempo médio por categoria, apontamos os gargalos e as categorias que precisam de atenção, seja pela dificuldade de transporte ou por outros problemas.

### 3.2 Mediana
**Demanda 5: vendas de eletrônicos no Nordeste.** As vendas de **Eletrônicos** no Nordeste cresceram e a equipe quer entender a distribuição dos valores. A média pode ser influenciada por compras muito altas ou muito baixas: qual medida retrata melhor o consumidor típico?

A **mediana** ($M_d$) é o valor que divide o conjunto exatamente ao meio:
1. Ordenar o conjunto de dados;
2. Identificar o número de observações $n$;
3. Identificar o elemento mediano:

$$ Elemento_{Md} = \left\{\begin{matrix}
\frac{n+1}2 & ,se \ n \ for \ ímpar \\
 &  \\
\frac{n}2 & ,se \ n \ for \ par \\
\end{matrix}\right.$$

4. Obter a mediana:

$$ M_d = \left\{\begin{matrix}
X_{Elemento_{Md}} & ,se \ n \ for \ ímpar \\
 &  \\
\frac{X_{Elemento_{Md}} + X_{Elemento_{Md}+1}}2 & ,se \ n \ for \ par \\
\end{matrix}\right.$$

**Usando a fórmula:**

In [ ]:
dados_nordeste = df[(df['regiao_cliente'] == 'Nordeste') & (df['categoria_produto'] == 'Eletrônicos')]
dados_nordeste = dados_nordeste.sort_values('total_compra')

valores = dados_nordeste['total_compra'].reset_index(drop=True)
n = len(valores)
if n % 2 == 1:
    mediana_formula = valores[(n + 1) // 2 - 1]
else:
    mediana_formula = (valores[n // 2 - 1] + valores[n // 2]) / 2
mediana_formula

**Usando a função `median()`**, comparada com a média:

In [ ]:
print('Mediana:', dados_nordeste['total_compra'].median())
print('Média:', round(dados_nordeste['total_compra'].mean(), 2))

**Por que usar a mediana?** A média é influenciada por valores extremos; a mediana reflete o ponto central das vendas e é mais adequada para dados assimétricos. Metade dos clientes gasta até cerca de R\$ 2.990,00 em eletrônicos, enquanto a média (R\$ 3.970,00) é puxada para cima por vendas muito altas.

**Visualizando a diferença:** o **histograma** é um gráfico de barras de uma distribuição de frequências: a base de cada barra representa uma classe e a altura, a sua frequência absoluta.

In [ ]:
sns.histplot(dados_nordeste['total_compra'], bins=30)
plt.axvline(dados_nordeste['total_compra'].mean(), color='red', linestyle='--', label='Média')
plt.axvline(dados_nordeste['total_compra'].median(), color='green', linestyle='-.', label='Mediana')
plt.legend()
plt.show()

Com uma diferença de quase R\$ 1.000,00 entre média e mediana, a mediana é a métrica mais confiável para o valor esperado de compra de eletrônicos no Nordeste. Isso apoia estratégias como promoções na faixa da mediana (consumidor típico) e campanhas de produtos de alto valor para quem gasta mais.

### 3.3 Moda
**Demanda 6: campanha promocional de livros.** Para a campanha "Pague X e Leve Y Livros", a equipe quer basear o pacote na **quantidade de livros mais frequentemente comprada**.

A **moda** ($M_o$) é o valor mais frequente de um conjunto de dados. É muito usada para dados qualitativos, mas também se aplica a dados numéricos.

In [ ]:
livros = df[df['categoria_produto'] == 'Livros']
livros['quantidade'].mode()

**Cuidados com a moda:** em dados numéricos pode haver mais de uma moda (multimodalidade). Outra forma de obtê-la é contar as ocorrências e filtrar as de frequência máxima, verificando se há uma ou mais modas.

In [ ]:
freq_livros = livros['quantidade'].value_counts()
freq_livros[freq_livros == freq_livros.max()]

Há **2 modas** na quantidade de livros vendidos (**bimodal**). Cabe à empresa decidir a campanha: para menos livros, para mais livros, ou duas campanhas (por exemplo "Pague 2 e Leve 3" e "Pague 4 e Leve 6"), verificando qual converte mais.

### 3.4 Relação entre média, mediana e moda
**Demanda 7: avaliações e tempo de entrega.** A hipótese é que clientes mais satisfeitos (nota 5) têm tempo de entrega mais curto. Vamos (1) calcular as medidas e ver a curva de densidade do tempo de entrega e (2) repetir para as avaliações nota 5.

A relação entre as medidas indica se a variável é simétrica ou assimétrica, se precisa de transformação ou se tende a ter outliers:

![relação entre média, mediana e moda](https://github.com/afonsosr2/estatistica-r-frequencias-medidas/blob/main/imagens/outros/relacao_media_mediana_moda.png?raw=true)

In [ ]:
def histograma_mtc(serie, titulo):
    """Histograma com curva de densidade e linhas de média, mediana e moda."""
    medidas = {'média': serie.mean(), 'mediana': serie.median(), 'moda': serie.mode()[0]}
    plt.figure(figsize=(10, 6))
    sns.histplot(serie, discrete=True, kde=True, stat='density', alpha=0.3)
    plt.axvline(medidas['média'], color='red', linestyle='--', label=f"Média: {medidas['média']:.2f}")
    plt.axvline(medidas['mediana'], color='green', linestyle='-.', label=f"Mediana: {medidas['mediana']:.2f}")
    plt.axvline(medidas['moda'], color='black', linestyle=':', label=f"Moda: {medidas['moda']:.2f}")
    plt.title(titulo)
    plt.legend()
    plt.show()
    return medidas


histograma_mtc(df['tempo_entrega'], 'Tempo de entrega - todas as vendas')

O tempo de entrega segue uma **distribuição normal simétrica**. A hipótese é que, para as vendas com nota 5, a curva seja **assimétrica à direita**, sugerindo que entregas mais rápidas estão ligadas a maior satisfação.

In [ ]:
nota_alta = df[df['avaliacao'] == 5]
histograma_mtc(nota_alta['tempo_entrega'], 'Tempo de entrega - avaliações nota 5')

Os dados confirmam que entregas mais rápidas estão associadas a notas 5, embora algumas entregas mais longas ainda recebam notas altas. Marketing e logística podem otimizar rotas para reduzir o tempo médio de entrega e usar a promessa de entrega rápida em campanhas.

## 4. Investigando os dados dos colaboradores
Análises da remuneração dos colaboradores para o RH e o financeiro: salários, desempenho e equidade salarial entre cargos e gênero.

### 4.1 Histograma de salários com amplitude fixa
**Demanda 8:** investigar a distribuição dos salários para identificar faixas salariais e a concentração de colaboradores em cada intervalo (ajuste de benefícios, disparidades salariais e retenção de talentos).

O número de classes vem da **regra de Sturges**:

$$k = 1 + \frac {10}{3}\log_{10}n$$

onde $k$ = número de classes e $n$ = número de observações.

In [ ]:
url = 'https://raw.githubusercontent.com/alura-cursos/Estatisticas-Python-frequencias-medidas/refs/heads/main/dados/colaboradores.csv'
colaboradores = pd.read_csv(url)

n = len(colaboradores)
k = int(1 + (10 / 3) * math.log10(n))
k

In [ ]:
faixas = colaboradores.copy()
faixas['faixa_salarial'] = pd.cut(faixas['remuneracao'], bins=k, include_lowest=True)

faixas_salariais = faixas.groupby('faixa_salarial', observed=False).size().reset_index(name='frequencia')
faixas_salariais['porcentagem'] = round(faixas_salariais['frequencia'] / faixas_salariais['frequencia'].sum() * 100, 2)
faixas_salariais

Mais da metade dos colaboradores (52,5%) está na 1ª faixa (de R\$ 1.890,00 até R\$ 4.080,00).

In [ ]:
sns.histplot(bins=k, data=colaboradores, x='remuneracao', kde=True)
plt.show()

A distribuição é assimétrica à direita, como esperado para salários, e mais de 95% das remunerações vão até R\$ 10.000,00.

### 4.2 Medidas separatrizes
**Demanda 9:** entender a distribuição dos salários em faixas e verificar se a remuneração das pessoas coordenadoras está entre o 1% de maiores salários.

**Quartis, decis e percentis** são medidas de posição semelhantes à mediana: os **quartis** dividem os dados em 4 partes iguais (úteis para faixas), os **decis** em 10 e os **percentis** em 100 (análises granulares, como os 5% ou 1% maiores valores).

In [ ]:
quartis = {
    'Q1': colaboradores['remuneracao'].quantile(0.25),
    'Q2': colaboradores['remuneracao'].quantile(0.50),
    'Q3': colaboradores['remuneracao'].quantile(0.75)
}
quartis

In [ ]:
plt.figure(figsize=(15, 6))
sns.histplot(binwidth=500, data=colaboradores, x='remuneracao')
for q in quartis.values():
    plt.axvline(q, color='red', linestyle='dashed')
plt.show()

Por causa dos altos salários dos cargos gerenciais, a remuneração tem **distribuição assimétrica**: muitos recebem um valor base e poucos, salários altos.

Agora, as pessoas coordenadoras (3º nível hierárquico) estão entre o 1% de maiores salários?

In [ ]:
percentil_99 = colaboradores['remuneracao'].quantile(0.99)

coordenadores = colaboradores[colaboradores['cargo'] == 'Coordenador(a)']
coordenadores_alta_remuneracao = coordenadores[coordenadores['remuneracao'] > percentil_99]

print('Percentil 99:', percentil_99)
print('Coordenadores(as):', len(coordenadores))
print('Coordenadores(as) acima do percentil 99:', len(coordenadores_alta_remuneracao))

As pessoas coordenadoras estão entre o 1% com maiores salários. O RH pode verificar se isso segue as políticas salariais e definir políticas mais transparentes e alinhadas ao mercado.

### 4.3 Classificando dados pelas medidas separatrizes
**Demanda 10:** identificar os **20% mais novos** para um programa de aceleração profissional.

In [ ]:
plt.figure(figsize=(15, 6))
sns.histplot(data=colaboradores, x='idade', bins=10, cumulative=True, stat='proportion', kde=True)
plt.axhline(0.20, color='red', linestyle='dashed')
plt.show()

In [ ]:
# 1. Ordenar pela idade
idades_classificacao = colaboradores.sort_values(by='idade').copy()

# 2. Posição relativa acumulada de cada linha
idades_classificacao['cumulativo'] = np.arange(1, len(idades_classificacao) + 1) / len(idades_classificacao)

# 3. Qualificados: os 20% primeiros
idades_classificacao['qualificado'] = idades_classificacao['cumulativo'] <= 0.20

idades_qualificados = idades_classificacao[idades_classificacao['qualificado']]
print('Qualificados:', len(idades_qualificados))
print('Idades:', sorted(idades_qualificados['idade'].unique()))

Há **2.218 pessoas** qualificadas, os **20% mais jovens**, público ideal para o programa, pois estão no início da carreira.

### 4.4 Boxplot
**Demanda 11:** investigar as diferenças salariais entre os sexos biológicos, para reforçar a equidade salarial.

![boxplot](https://github.com/afonsosr2/estatistica-r-frequencias-medidas/blob/main/imagens/outros/boxplot.png?raw=true)

O **boxplot** mostra a distribuição de dados quantitativos e facilita comparações entre grupos:
- **Caixa:** intervalo interquartil (IIQ), de Q1 (25%) a Q3 (75%);
- **Linha mediana:** a mediana (Q2);
- **Whiskers ("bigodes"):** variabilidade fora do IIQ, geralmente até 1,5 IIQ a partir dos quartis;
- **Outliers:** pontos fora dos whiskers.

In [ ]:
plt.figure(figsize=(8, 6))
sns.boxplot(x=colaboradores['remuneracao'], color='steelblue')
plt.title('Boxplot de Salários')
plt.xlabel('Salário (R$)')
plt.show()

Os valores de cada elemento do boxplot podem ser obtidos com os quartis ou com `describe()`:

In [ ]:
colaboradores['remuneracao'].describe()

In [ ]:
plt.figure(figsize=(8, 6))
sns.boxplot(x='remuneracao', y='sexo_biologico', hue='sexo_biologico', data=colaboradores)
plt.title('Boxplot de Salários')
plt.xlabel('Salário (R$)')
plt.show()

O sexo masculino parece ter distribuição salarial maior, mas os outliers dificultam a leitura. Como mais de 95% das remunerações vão até R\$ 10.000,00, filtramos por esse valor:

In [ ]:
colaboradores_filtrados = colaboradores[colaboradores['remuneracao'] <= 10e3]

plt.figure(figsize=(8, 6))
sns.boxplot(x='remuneracao', y='sexo_biologico', data=colaboradores_filtrados, hue='sexo_biologico')
plt.title('Boxplot de Salários por Sexo Biológico')
plt.xlabel('Salário (R$)')
plt.ylabel('Sexo Biológico')
plt.show()

**A mediana dos salários do sexo feminino é menor** (abaixo de R\$ 4.000,00) que a do masculino (acima desse valor). As estatísticas descritivas por grupo reforçam a análise:

In [ ]:
estat_rem = colaboradores.groupby('sexo_biologico')['remuneracao'].agg(
    q1=lambda x: x.quantile(0.25),
    mediana='median',
    media='mean',
    q3=lambda x: x.quantile(0.75),
    IIQ=lambda x: x.quantile(0.75) - x.quantile(0.25),
    desvio_padrao='std',
    minimo='min',
    maximo='max',
    contagem='count'
).reset_index()
estat_rem

O RH deve investigar as causas, considerando cargos similares, e revisar políticas salariais levando em conta cargo, experiência e desempenho.

## 5. Analisando as variações dos dados
As **medidas de dispersão** indicam o grau de variação dos valores de uma variável quantitativa em relação à sua média. Medidas de tendência central e de posição podem não bastar quando os dados são muito dispersos.

### 5.1 Desvio médio absoluto
**Demanda 12 (parte 1):** há feedbacks de falta de padrão entre os salários de **Inteligência comercial** e **Consultor(a) de vendas**. Média e mediana não bastam para avaliar a uniformidade dos salários.

O **desvio médio absoluto (MAD)** é a média das diferenças absolutas entre cada valor e a média:

$$MAD = \frac 1n\sum_{i=1}^{n}|X_i-\bar{X}|$$

onde $n$ = número de observações, $X_i$ = i-ésimo valor e $\bar{X}$ = média da distribuição.

In [ ]:
cargos = colaboradores[colaboradores['cargo'].isin(['Inteligência comercial', 'Consultor(a) de vendas'])]
cargos.groupby('cargo')['remuneracao'].describe()[['mean', '50%']]

Médias e medianas bem próximas, mas elas não representam a dispersão. Desvios de uma amostra de 10 salários em relação à média:

In [ ]:
amostra = cargos.sample(n=10, random_state=42)

plt.stem(amostra['remuneracao'], bottom=amostra['remuneracao'].mean())
plt.axhline(y=amostra['remuneracao'].mean(), color='red')
plt.show()

In [ ]:
estat_colab = cargos.groupby('cargo')['remuneracao'].agg(
    media='mean',
    mediana='median',
    mad=lambda x: (abs(x - x.mean())).mean()
).reset_index()
estat_colab

**Visualizando graficamente:**

In [ ]:
plt.figure(figsize=(8, 6))
sns.boxplot(x='remuneracao', y='cargo', data=cargos)
plt.title('Distribuição de Salários por Cargo')
plt.xlabel('Salário (R$)')
plt.ylabel('')
plt.show()

Média e mediana são muito próximas entre os dois cargos, com variação ligeiramente diferente entre eles, vista pelo MAD e pelo intervalo interquartil. O RH pode investigar os critérios salariais (o esperado era o mesmo nível) e padronizar políticas para cargos equivalentes.

### 5.2 Variância e desvio padrão
**Demanda 12 (parte 2):** possível falta de padrão entre os salários de **Estoquista** e **Repositor(a)**, os cargos com mais funcionários. Qual apresenta maior inconsistência?

A **variância** ($\sigma^2$ ou $S^2$) mostra o quão distante cada valor está da média: os desvios em torno da média são elevados ao quadrado.

**Variância populacional:** $$\sigma^2 = \frac 1n\sum_{i=1}^{n}(X_i-\mu)^2$$

**Variância amostral:** $$S^2 = \frac 1{n-1}\sum_{i=1}^{n}(X_i-\bar{X})^2$$

Como a variância fica no quadrado da unidade original, usamos também o **desvio padrão** ($\sigma$ ou $S$), sua raiz quadrada, que também mede a confiança de uma estimativa:

**Desvio padrão populacional:** $$\sigma = \sqrt{\frac 1n\sum_{i=1}^{n}(X_i-\mu)^2} \Longrightarrow \sigma = \sqrt{\sigma^2}$$

**Desvio padrão amostral:** $$S = \sqrt{\frac 1{n-1}\sum_{i=1}^{n}(X_i-\bar{X})^2} \Longrightarrow S = \sqrt{S^2}$$

In [ ]:
operacional = colaboradores[colaboradores['cargo'].isin(['Estoquista', 'Repositor(a)'])]

estat_operacional = operacional.groupby('cargo')['remuneracao'].agg(
    media='mean',
    mediana='median',
    variancia='var',
    desv_pad='std',
    desvio_abs_medio=lambda x: (abs(x - x.mean())).mean()
).reset_index()
estat_operacional

`np.var`/`np.std` calculam as medidas **populacionais** (divisor $n$); os métodos `.var()`/`.std()` do pandas, as **amostrais** ($n-1$):

In [ ]:
print('np.var  (populacional):', np.var(operacional['remuneracao']))
print('.var()  (amostral):    ', operacional['remuneracao'].var())
print('np.std  (populacional):', np.std(operacional['remuneracao']))
print('.std()  (amostral):    ', operacional['remuneracao'].std())

1. **Média e mediana:** valores próximos nos dois cargos, indicando simetria na centralidade.
2. **Variância e desvio padrão:** maiores para Estoquista, revelando maior dispersão nos salários.
3. **MAD vs. desvio padrão:** o MAD é menor que o desvio padrão nos dois casos, indicando influência de valores extremos.

In [ ]:
sns.violinplot(data=operacional, x='cargo', y='remuneracao')
plt.title('Distribuição de Salários por Cargo')
plt.xlabel('Cargo')
plt.ylabel('Salário (R$)')
plt.show()

Estoquista tem maior variabilidade salarial. O RH deve investigar se a inconsistência reflete discrepâncias injustas ou justificativas de mérito/experiência, e ajustar as políticas para promover equidade.

### 5.3 Outras medidas: amplitude, erro padrão e coeficiente de variação
- **Amplitude:** diferença entre o maior e o menor valor.
- **Erro padrão:** $EP = \frac{S}{\sqrt{n}}$, a variabilidade esperada da média amostral.
- **Coeficiente de variação:** $CV = \frac{S}{\bar{X}} \times 100$, a dispersão relativa à média, em %.

In [ ]:
amplitude = operacional['remuneracao'].max() - operacional['remuneracao'].min()
desvio_padrao = operacional['remuneracao'].std()
erro_padrao = desvio_padrao / np.sqrt(len(operacional))
coeficiente_de_variacao = (desvio_padrao / operacional['remuneracao'].mean()) * 100

print('Amplitude:', amplitude)
print('Erro padrão:', erro_padrao)
print(f'Coeficiente de variação: {coeficiente_de_variacao:.2f}%')

## 6. Desafios
Exercícios "Mão na Massa" do curso, organizados por aula: análise descritiva de uma adaptação da **PNAD 2015** (pessoas responsáveis pelo domicílio).

**Fonte:** https://www.ibge.gov.br/estatisticas/sociais/populacao/9127-pesquisa-nacional-por-amostra-de-domicilios.html?edicao=9128

**Variáveis:** `UF` (unidade federativa); `Sexo` (0 Masculino, 1 Feminino); `Idade` (anos); `Cor` (0 Indígena, 2 Branca, 4 Preta, 6 Amarela, 8 Parda); `Anos.de.Estudo` (1 Sem instrução e menos de 1 ano ... 16 = 15 anos ou mais, 17 Não determinados); `Renda` (rendimento mensal do trabalho principal); `Altura` (metros, dado sintético).

> Nos dados originais foram eliminados os registros com Renda inválida (999 999 999 999) ou ausente, e mantidas só as **pessoas de referência** de cada domicílio.

### 6.1 Aula 01 - Explorando novos dados
Importar o dataset, ler as informações (linhas, colunas, tipos) e explorar a variável `UF`.

In [ ]:
url = 'https://raw.githubusercontent.com/alura-cursos/Estatisticas-Python-frequencias-medidas/refs/heads/main/dados/dados_desafio.csv'
dados = pd.read_csv(url)
dados.info()

In [ ]:
dados.head()

In [ ]:
UF = dados['UF'].value_counts().reset_index()
sns.barplot(data=UF, x='count', y='UF')
plt.show()

Transformar `Sexo`, `Cor` e `Anos.de.Estudo` em `Categorical` (ordenadas) e trocar os códigos pelos rótulos:

In [ ]:
sexo = {0: 'Masculino', 1: 'Feminino'}

cor = {0: 'Indígena', 2: 'Branca', 4: 'Preta', 6: 'Amarela', 8: 'Parda'}

anos_de_estudo = {1: 'Sem instrução e menos de 1 ano', 2: '1 ano', 3: '2 anos', 4: '3 anos', 5: '4 anos', 6: '5 anos',
                  7: '6 anos', 8: '7 anos', 9: '8 anos', 10: '9 anos', 11: '10 anos', 12: '11 anos', 13: '12 anos',
                  14: '13 anos', 15: '14 anos', 16: '15 anos ou mais', 17: 'Não determinados'}

In [ ]:
dados['Cat.Sexo'] = pd.Categorical(dados['Sexo'], categories=[0, 1], ordered=True)
dados['Cat.Sexo'] = dados['Cat.Sexo'].map(sexo)

dados['Cat.Cor'] = pd.Categorical(dados['Cor'], categories=[0, 2, 4, 6, 8], ordered=True)
dados['Cat.Cor'] = dados['Cat.Cor'].map(cor)

dados['Cat.Anos.Estudo'] = pd.Categorical(dados['Anos.de.Estudo'], categories=range(1, 18), ordered=True)
dados['Cat.Anos.Estudo'] = dados['Cat.Anos.Estudo'].map(anos_de_estudo)

dados[['Anos.de.Estudo', 'Cat.Anos.Estudo']].drop_duplicates().sort_values('Anos.de.Estudo')

In [ ]:
anos_estudo = dados['Anos.de.Estudo'].value_counts().reset_index()
sns.barplot(data=anos_estudo, x='Anos.de.Estudo', y='count')
plt.show()

Menor e maior `Renda`, em texto (f-string):

In [ ]:
print(f"A menor Renda da base de dados é de R$ {dados['Renda'].min():,.2f} e a maior é de R$ {dados['Renda'].max():,.2f}")

### 6.2 Aula 02 - Perfil das pessoas responsáveis
Tabela de frequências por `Cat.Sexo`, com percentual:

In [ ]:
tb_sexo = dados['Cat.Sexo'].value_counts().reset_index(name='frequencia')
tb_sexo['freq. relativa'] = round(tb_sexo['frequencia'] / tb_sexo['frequencia'].sum() * 100, 2)
tb_sexo

Frequências absolutas e relativas cruzando `Cat.Sexo` e `Cat.Cor`:

In [ ]:
tb_sexo_cor = pd.crosstab(dados['Cat.Sexo'], dados['Cat.Cor'])
tb_sexo_cor

In [ ]:
tb_sexo_cor_rel = round(pd.crosstab(dados['Cat.Sexo'], dados['Cat.Cor'], normalize=True) * 100, 2)
tb_sexo_cor_rel

`Renda` média por `Cat.Sexo` e `Cat.Cor`:

In [ ]:
renda_media_sexo_cor = round(pd.crosstab(dados['Cat.Sexo'], dados['Cat.Cor'], values=dados['Renda'], aggfunc='mean'), 2)
renda_media_sexo_cor

### 6.3 Aula 03 - Tendência central da renda
Média, mediana e moda da `Renda` (a moda é verificada contando a frequência dos valores modais):

In [ ]:
moda_renda = dados['Renda'].mode()
tendencia_central = {
    'media': round(dados['Renda'].mean(), 2),
    'mediana': dados['Renda'].median(),
    'moda': moda_renda.tolist()
}
print(tendencia_central)
dados.loc[dados['Renda'].isin(moda_renda), 'Renda'].value_counts()

Top 5 estados pela média de `Renda`:

In [ ]:
renda_estados = round(dados.groupby('UF')['Renda'].mean().reset_index(), 2).sort_values(by='Renda', ascending=False)
sns.barplot(data=renda_estados.head(), x='Renda', y='UF')
plt.show()

Média, mediana e máximo da `Renda` por estado da Região Sudeste e por `Cat.Sexo`:

In [ ]:
dados_sudeste = dados[dados['UF'].isin(['São Paulo', 'Rio de Janeiro', 'Espírito Santo', 'Minas Gerais'])]

media_sudeste = pd.crosstab(dados_sudeste['UF'], dados_sudeste['Cat.Sexo'], values=dados_sudeste['Renda'], aggfunc='mean')
media_sudeste

In [ ]:
mediana_sudeste = pd.crosstab(dados_sudeste['UF'], dados_sudeste['Cat.Sexo'], values=dados_sudeste['Renda'], aggfunc='median')
mediana_sudeste

In [ ]:
renda_max_sudeste = pd.crosstab(dados_sudeste['UF'], dados_sudeste['Cat.Sexo'], values=dados_sudeste['Renda'], aggfunc='max')
renda_max_sudeste

Histogramas com curva de densidade de `Altura` e `Idade`, comparados com as medidas de tendência central (MTC):

In [ ]:
sns.histplot(bins=15, data=dados, x='Altura', kde=True, kde_kws={'bw_adjust': 2})
plt.title('Distribuição das alturas das pessoas responsáveis pelo domicílio')
plt.xlabel('Altura (metros)')
plt.ylabel('Frequência')
plt.show()

{'media': round(dados['Altura'].mean(), 2),
 'mediana': dados['Altura'].median(),
 'moda': dados['Altura'].mode()[0]}

In [ ]:
sns.histplot(bins=15, data=dados, x='Idade', kde=True, kde_kws={'bw_adjust': 2})
plt.title('Distribuição das idades das pessoas responsáveis pelo domicílio')
plt.xlabel('Idade (anos)')
plt.ylabel('Frequência')
plt.show()

{'media': round(dados['Idade'].mean(), 2),
 'mediana': dados['Idade'].median(),
 'moda': dados['Idade'].mode()[0]}

### 6.4 Aula 04 - Rendas e suas divisões
Número de classes para `Renda` até R\$ 15.000 pela regra de Sturges e histograma com essas classes.

> O limite inferior da 1ª faixa pode sair negativo: `pd.cut` estende o intervalo mínimo em $min(Renda) - (max(Renda) - min(Renda)) \times 0.001$.

In [ ]:
renda_15000 = dados[dados['Renda'] <= 15000].copy()

n = len(renda_15000)
k = int(1 + (10 / 3) * math.log10(n))
print('n =', n, '| k =', k)

renda_15000['faixa_salarial'] = pd.cut(renda_15000['Renda'], bins=k, include_lowest=True)
renda_15000.groupby('faixa_salarial', observed=False).size().reset_index(name='frequencia')

In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(data=renda_15000, x='Renda', bins=k, kde=True)
plt.show()

**1. Percentual de pessoas responsáveis que ganhava até um salário mínimo em 2015 (R\$ 788,00):**

In [ ]:
round(len(dados[dados['Renda'] <= 788]) / len(dados) * 100, 2)

**2. Renda máxima de 95% das pessoas responsáveis:**

In [ ]:
int(dados['Renda'].quantile(0.95))

**3. Renda mínima do 1% mais bem pago:**

In [ ]:
pct99 = dados['Renda'].quantile(0.99)
dados.loc[dados['Renda'] >= pct99, 'Renda'].min()

**4. Renda máxima de 25%, 50% e 75% de quem recebeu até R\$ 6.000, com boxplot:**

In [ ]:
renda_ate_6000 = dados[dados['Renda'] <= 6000]
for q in [0.25, 0.50, 0.75]:
    print(f"Renda máxima de {q:.0%}: {renda_ate_6000['Renda'].quantile(q)}")

plt.figure(figsize=(10, 5))
sns.boxplot(data=renda_ate_6000, x='Renda')
plt.show()

Boxplot da `Renda` até o percentil 95 (R\$ 6.000) por `Cat.Cor` e `Cat.Sexo` (terceira variável no parâmetro `hue`):

In [ ]:
plt.figure(figsize=(10, 5))
sns.boxplot(data=renda_ate_6000, x='Renda', y='Cat.Cor', hue='Cat.Sexo')
plt.show()

**Idade limite para 20% da população:** histograma acumulado com curva de densidade, idade limite e quantas pessoas se encaixam.

In [ ]:
sns.histplot(data=dados, x='Idade', bins=10, cumulative=True, stat='proportion', kde=True)
plt.axhline(0.20, color='red', linestyle='dashed')
plt.show()

In [ ]:
idades = dados.sort_values(by='Idade').copy()
idades['cumulativo'] = np.arange(1, len(idades) + 1) / len(idades)
idades['qualificado'] = idades['cumulativo'] <= 0.20

qualificados = idades[idades['qualificado']]
print('Pessoas nos 20% mais jovens:', len(qualificados))
qualificados.tail(1)

In [ ]:
sns.boxplot(data=qualificados, x='Idade', y='Cat.Sexo', hue='Cat.Sexo')
plt.show()

### 6.5 Aula 05 - Variação da renda
**1. Desvio médio absoluto, variância e desvio padrão da renda:**

In [ ]:
{
    'desvio_medio_abs': (dados['Renda'] - dados['Renda'].mean()).abs().mean(),
    'variancia': dados['Renda'].var(),
    'desvio_padrao': dados['Renda'].std()
}

**2. Média, mediana e desvio padrão da renda por anos de estudo:**

In [ ]:
renda_estudo = dados.groupby('Cat.Anos.Estudo', observed=False)['Renda'].agg(['mean', 'median', 'std']).reset_index()
renda_estudo.columns = ['Anos de Estudo', 'Média', 'Mediana', 'Desvio Padrão']
renda_estudo

**3. Média, mediana e desvio padrão da renda por sexo, para rendas até R\$ 15.000:**

In [ ]:
renda_15000.groupby('Cat.Sexo', observed=False)['Renda'].agg(['mean', 'median', 'std']).reset_index()

**4. Região Centro-Oeste:** boxplot da `Renda` até R\$ 10.000 por `Cat.Sexo` e tabela com média, mediana e desvio padrão.

In [ ]:
centro_oeste = dados[dados['UF'].isin(['Mato Grosso', 'Mato Grosso do Sul', 'Goiás', 'Distrito Federal'])]
print('Registros no Centro-Oeste:', len(centro_oeste))

centro_oeste_10k = centro_oeste[centro_oeste['Renda'] <= 10000]

plt.figure(figsize=(10, 5))
sns.boxplot(data=centro_oeste_10k, x='Renda', y='Cat.Sexo', hue='Cat.Sexo')
plt.show()

In [ ]:
centro_oeste_10k.groupby('Cat.Sexo', observed=False)['Renda'].agg(media='mean', mediana='median', desvio_padrao='std')